In [16]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [17]:
data=pd.read_csv('Churn_Modelling.csv')
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
labe_encoder_gender=LabelEncoder()
data['Gender']=labe_encoder_gender.fit_transform(data['Gender'])
onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
X=data.drop('Exited',axis=1)
y=data['Exited']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(labe_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [18]:
from tensorflow.keras.layers import Input
def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons,activation='relu'))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
    return model

In [19]:
model=KerasClassifier(layers=1,neurons=32,model=create_model,epochs=50,batch_size=10,verbose=0)

In [22]:
param_grid={
    'neurons':[32,64],
    'layers':[1,2],
    'epochs':[50]
}

In [24]:
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=1,cv=3,verbose=2)
grid_result=grid.fit(X_train,y_train)
print("Best:", grid_result.best_score_)
print("Params:", grid_result.best_params_)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END ....................epochs=50, layers=1, neurons=32; total time= 1.1min
[CV] END ....................epochs=50, layers=1, neurons=32; total time= 1.2min
[CV] END ....................epochs=50, layers=1, neurons=32; total time= 1.2min
[CV] END ....................epochs=50, layers=1, neurons=64; total time= 1.1min
[CV] END ....................epochs=50, layers=1, neurons=64; total time=  58.3s
[CV] END ....................epochs=50, layers=1, neurons=64; total time=  59.0s
[CV] END ....................epochs=50, layers=2, neurons=32; total time= 1.0min
[CV] END ....................epochs=50, layers=2, neurons=32; total time= 1.0min
[CV] END ....................epochs=50, layers=2, neurons=32; total time= 1.0min
[CV] END ....................epochs=50, layers=2, neurons=64; total time= 1.0min
[CV] END ....................epochs=50, layers=2, neurons=64; total time= 1.0min
[CV] END ....................epochs=50, layers=2,